# Scenario impact analysis

Use **`apply_scenario_and_revalue`** to stress a **`PortfolioSpec`** against a **`MarketContext`** snapshot, then compare base vs stressed **`total_base_ccy`**. A simple **P&L decomposition** uses **single-factor** scenarios (rates-only, credit-only, FX-only) as *attribution proxies*—not a formal risk model.

## Concept

- **`value_portfolio(spec_json, market_json)`** returns **`PortfolioValuation`** JSON; read **`total_base_ccy.amount`** (string decimal) for base-currency PV.
- **`apply_scenario_and_revalue(spec, scenario_json, market_json)`** returns **(valuation_json, report_json)** after applying the scenario engine to the market used for repricing.
- **P&L decomposition (conceptual)**:
  - **Rates contribution**: shock **discount** curves (**USD-OIS** and **EUR-OIS** in the code cell). EUR discount bumps use the engine’s **synthetic deposit** re-bootstrap; the index used there is the registered **€STR / ESTR-OIS** family (**`EUR-ESTR-OIS`** in conventions), not a separate forward curve in this snapshot.
  - **Credit contribution**: shock **`par_cds` / hazard-linked** curves.
  - **FX contribution**: **`market_fx_pct`** on FX pairs affecting foreign-currency positions.

Summing single-factor deltas **overlaps** (non-linear interactions); treat the sum as an **approximate split** for storytelling, not additive risk decomposition.

In [ ]:
import json
from datetime import date

from finstack.core.market_data import DiscountCurve, HazardCurve, MarketContext
from finstack.portfolio import value_portfolio, apply_scenario_and_revalue
from finstack.scenarios import list_builtin_templates
from finstack.scenarios import build_scenario_spec, validate_scenario_spec

AS_OF = "2025-01-15"
bd = date(2025, 1, 15)

usd = DiscountCurve("USD-OIS", bd, [(0.0, 1.0), (0.5, 0.996), (1.0, 0.988), (5.0, 0.92)])
eur = DiscountCurve("EUR-OIS", bd, [(0.0, 1.0), (0.5, 0.995), (1.0, 0.986), (5.0, 0.90)])
hz = HazardCurve("CORP-HAZARD", bd, [(0.0, 0.0), (1.0, 0.016), (5.0, 0.022)], recovery_rate=0.4)
mc = MarketContext().insert(usd).insert(eur).insert(hz)
state = json.loads(mc.to_json())
state["fx"] = {
    "config": {"pivot_currency": "USD", "enable_triangulation": False, "cache_capacity": 256},
    "quotes": [["EUR", "USD", 1.08]],
}
market_json = json.dumps(state)
print("Market ready: USD-OIS, EUR-OIS, CORP-HAZARD, EUR/USD=1.08")
print("Built-in scenario templates available:", len(list_builtin_templates()))

portfolio = json.dumps(
    {
        "id": "attrib-demo",
        "as_of": AS_OF,
        "base_ccy": "USD",
        "entities": {"E1": {"id": "E1"}},
        "positions": [
            {
                "position_id": "P-USD",
                "entity_id": "E1",
                "instrument_id": "DEP-USD",
                "instrument_spec": {
                    "type": "deposit",
                    "spec": {
                        "id": "DEP-USD",
                        "notional": {"amount": 1_000_000.0, "currency": "USD"},
                        "start_date": AS_OF,
                        "maturity": "2025-10-15",
                        "day_count": "Act360",
                        "quote_rate": 0.052,
                        "discount_curve_id": "USD-OIS",
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            },
            {
                "position_id": "P-EUR",
                "entity_id": "E1",
                "instrument_id": "DEP-EUR",
                "instrument_spec": {
                    "type": "deposit",
                    "spec": {
                        "id": "DEP-EUR",
                        "notional": {"amount": 400_000.0, "currency": "EUR"},
                        "start_date": AS_OF,
                        "maturity": "2025-10-15",
                        "day_count": "Act360",
                        "quote_rate": 0.038,
                        "discount_curve_id": "EUR-OIS",
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            },
            {
                "position_id": "P-CDS",
                "entity_id": "E1",
                "instrument_id": "CDS-1",
                "instrument_spec": {
                    "type": "credit_default_swap",
                    "spec": {
                        "id": "CDS-1",
                        "notional": {"amount": 1_500_000.0, "currency": "USD"},
                        "side": "pay",
                        "convention": "isda_na",
                        "premium": {
                            "start": AS_OF,
                            "end": "2030-01-15",
                            "frequency": {"count": 3, "unit": "months"},
                            "stub": "ShortFront",
                            "bdc": "following",
                            "calendar_id": None,
                            "day_count": "Act360",
                            "spread_bp": 110.0,
                            "discount_curve_id": "USD-OIS",
                        },
                        "protection": {
                            "credit_curve_id": "CORP-HAZARD",
                            "recovery_rate": 0.4,
                            "settlement_delay": 3,
                        },
                        "pricing_overrides": {},
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            },
        ],
    }
)
print("Portfolio positions: USD deposit, EUR deposit, USD CDS on CORP-HAZARD")

base_val = value_portfolio(portfolio, market_json)
base_total = float(json.loads(base_val)["total_base_ccy"]["amount"])
print(f"Base total (USD): {base_total:,.2f}")

rates_ops = [
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-OIS", "bp": 45.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "EUR-OIS", "bp": 45.0},
]
rates_scn = build_scenario_spec("rates", json.dumps(rates_ops), "r", "")
credit_scn = build_scenario_spec("credit", json.dumps([{"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CORP-HAZARD", "discount_curve_id": "USD-OIS", "bp": 85.0}]), "c", "")
fx_scn = build_scenario_spec("fx", json.dumps([{"kind": "market_fx_pct", "base": "EUR", "quote": "USD", "pct": -3.0}]), "f", "")
print("Specs valid:", validate_scenario_spec(rates_scn), validate_scenario_spec(credit_scn), validate_scenario_spec(fx_scn))

def pv_total(val_json: str) -> float:
    return float(json.loads(val_json)["total_base_ccy"]["amount"])

single_dpl = []
for label, sj in [
    ("Rates +45bp USD-OIS & EUR-OIS (USD leg, EUR depo, CDS disc.)", rates_scn),
    ("Credit +85bp par CDS", credit_scn),
    ("FX -3% EURUSD (EUR weakens)", fx_scn),
]:
    vj, rep = apply_scenario_and_revalue(portfolio, sj, market_json)
    t = pv_total(vj)
    d = t - base_total
    single_dpl.append(d)
    print(f"{label}: total={t:,.2f}  dP&L={d:,.2f}  report_ops={json.loads(rep)['operations_applied']}")

sum_single_factors = sum(single_dpl)
print(f"Sum of single-factor dP&Ls (storytelling, not additive risk): {sum_single_factors:,.2f}")

# Combined stress
combined_ops = json.loads(rates_scn)["operations"] + json.loads(credit_scn)["operations"] + json.loads(fx_scn)["operations"]
combo = build_scenario_spec("all", json.dumps(combined_ops), "all", "")
vj2, rep2 = apply_scenario_and_revalue(portfolio, combo, market_json)
t2 = pv_total(vj2)
d_combo = t2 - base_total
print(f"Combined scenario total={t2:,.2f}  dP&L={d_combo:,.2f}")
cross_residual = d_combo - sum_single_factors
print(f"Cross-effect residual (combined dP&L minus sum of single-factor dP&L): {cross_residual:,.2f}")
print("Note: non-zero residual is expected (convexity, discounting interactions, FX/rates overlap).")


## Takeaways

- **`apply_scenario_and_revalue`** is the portfolio-level bridge: **scenario JSON → shocked market → repricing**.
- Read **`total_base_ccy.amount`** from valuation JSON for **before/after** PV in **base currency**.
- Use **single-factor** scenarios as a **storytelling decomposition**; the **combined** shock highlights **interaction** effects missing from the sum of parts.
- Ensure **FX quotes** exist in **`market_json`** when using **`market_fx_pct`**.
